### Dataset Sampling and Preprocessing

In this notebook, I sample and prepare our evaluation datasets from the Kaggle Resume dataset and the JobStreet Malaysia dataset:

1. **Sampling**:
   - **Information Technology (IT)**: 20 resumes and 40 job postings.
   - **Accounting**: 10 resumes and 20 job postings (used as comparison/cross-domain control).
   - In total: **30 Candidates** (`C01` to `C30`) and **60 Job Postings** (`J01` to `J60`).
2. **Adding Search Preferences**: The Kaggle resumes have detailed work experience and education, but lack job search preferences. I assign realistic preferences for the Malaysian market (target job title, location by state, expected salary range in MYR, and job type) based on each candidate's experience level.
3. **Cleaning Job Postings**: Remove duplicate jobs, drop very short descriptions (< 30 words), and parse salary text like `RM 3,000 - RM 5,000` into numeric min/max values.
4. **Export**: Save the cleaned and sampled datasets to `experiments/dataset/processed/` for ground truth annotation and matching experiments.


In [1]:
import os, sys, re
from pathlib import Path
import numpy as np
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    try: sys.stdout.reconfigure(encoding='utf-8')
    except: pass

RANDOM_STATE = 42

PROJECT_ROOT = Path("c:/Users/yzhoo/AI-Agent-Job-Search")
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
DATASET_DIR = EXPERIMENTS_DIR / "dataset"
RESUME_CSV_PATH = DATASET_DIR / "resume dataset" / "Resume" / "Resume.csv"
JOB_CSV_PATH = DATASET_DIR / "job_dataset" / "jobstreet_all_job_dataset.csv"
PROCESSED_DIR = DATASET_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Experiment paths and settings:")
print(f"Random State: {RANDOM_STATE}")
print(f"Resume dataset: {RESUME_CSV_PATH} (Exists: {RESUME_CSV_PATH.exists()})")
print(f"Job dataset:    {JOB_CSV_PATH} (Exists: {JOB_CSV_PATH.exists()})")
print(f"Output folder:  {PROCESSED_DIR}")
print("=" * 60)


EXPERIMENT ENVIRONMENT CONFIGURED
Random State: 42
Resume Dataset Path: c:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\resume dataset\Resume\Resume.csv (Exists: True)
Job Dataset Path:    c:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\job_dataset\jobstreet_all_job_dataset.csv (Exists: True)
Processed Output:    c:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\processed


### 1. Candidate Sampling and Preferences

The Kaggle resume dataset has 2,484 resumes across 24 categories.
1. The resumes have real work history, qualifications, and skills.
2. However, they do not include job preferences for the local job market (such as target role, preferred state in Malaysia, salary expectation in MYR, and job type).
3. Set up realistic job preferences for each candidate based on their actual experience tier and skills.


In [2]:
# Load Kaggle resume dataset
resumes_raw_df = pd.read_csv(RESUME_CSV_PATH)
print(f"Total resumes loaded: {len(resumes_raw_df):,d}")

# Filter out very short resumes (< 50 words)
resumes_raw_df["word_count"] = resumes_raw_df["Resume_str"].apply(lambda x: len(str(x).split()))
valid_resumes = resumes_raw_df[resumes_raw_df["word_count"] >= 50].copy()

# Filter by category (IT and Accounting)
it_resumes_pool = valid_resumes[valid_resumes["Category"] == "INFORMATION-TECHNOLOGY"].copy()
acc_resumes_pool = valid_resumes[valid_resumes["Category"] == "ACCOUNTANT"].copy()

print(f"Available IT resumes: {len(it_resumes_pool)}")
print(f"Available Accounting resumes: {len(acc_resumes_pool)}")

# Sample 20 IT candidates and 10 Accounting candidates
it_sampled = it_resumes_pool.sample(n=20, random_state=RANDOM_STATE).reset_index(drop=True)
acc_sampled = acc_resumes_pool.sample(n=10, random_state=RANDOM_STATE).reset_index(drop=True)

IT_SKILLS_LIST = [
    "Python", "Java", "C++", "C#", ".NET", "SQL", "MySQL", "PostgreSQL", "Oracle",
    "Linux", "Unix", "Windows Server", "Active Directory", "VMware", "Cloud", "AWS",
    "Azure", "GCP", "Docker", "Kubernetes", "DevOps", "CI/CD", "Git", "Networking",
    "Network Security", "TCP/IP", "Cisco", "Firewall", "Cybersecurity", "Information Security",
    "INFOSEC", "Information Assurance", "Penetration Testing", "SIEM", "Incident Response",
    "Disaster Recovery", "ITIL", "Agile", "Scrum", "Project Management", "Systems Analysis",
    "ERP", "SAP", "Help Desk", "Technical Support", "Troubleshooting", "Data Analysis",
    "Machine Learning", "Software Development", "API", "REST", "Web Development", "Database Administration"
]

ACC_SKILLS_LIST = [
    "Accounting", "Financial Reporting", "Financial Analysis", "General Ledger", "Accounts Payable",
    "Accounts Receivable", "Bank Reconciliation", "Payroll", "Taxation", "Tax Compliance",
    "Auditing", "Internal Audit", "External Audit", "Budgeting", "Forecasting", "Cash Flow",
    "Balance Sheet", "Profit and Loss", "GAAP", "IFRS", "SOX", "ERP", "SAP", "Oracle Financials",
    "QuickBooks", "Xero", "Excel", "VLOOKUP", "Pivot Tables", "Cost Accounting", "Variance Analysis",
    "Financial Statements", "Corporate Tax", "SST", "GST", "Internal Controls"
]

def extract_skills(text, category):
    keywords = IT_SKILLS_LIST if category == 'INFORMATION-TECHNOLOGY' else ACC_SKILLS_LIST
    text_lower = text.lower()
    found = []
    for kw in keywords:
        pattern = r'\b' + re.escape(kw.lower()) + r'\b'
        if re.search(pattern, text_lower):
            found.append(kw)
    return ", ".join(found[:10]) if found else "Domain Practical Competencies"

print(f"Sampled {len(it_sampled)} IT and {len(acc_sampled)} Accounting candidates.")


Total raw resumes loaded: 2,484
Valid IT candidate pool available: 120 resumes
Valid Accounting candidate pool available: 118 resumes
Candidate qualification heuristics initialized.


In [ ]:
# Candidate profile attributes: experience, education, and target salary (MYR)

audited_candidate_profiles = {
    # IT Candidates (C01 - C20)
    "C01": {"exp": 4, "tier": "Mid-Level (4-7 yrs)", "edu": "Associate", "s_min": 5500.0, "s_max": 8000.0},
    "C02": {"exp": 16, "tier": "Lead / Executive (15+ yrs)", "edu": "Bachelor's Degree", "s_min": 13500.0, "s_max": 19000.0},
    "C03": {"exp": 20, "tier": "Lead / Executive (15+ yrs)", "edu": "Bachelor's Degree", "s_min": 8500.0, "s_max": 13000.0},
    "C04": {"exp": 13, "tier": "Senior (8-14 yrs)", "edu": "Master's Degree", "s_min": 9500.0, "s_max": 14000.0},
    "C05": {"exp": 20, "tier": "Lead / Executive (15+ yrs)", "edu": "Master's Degree", "s_min": 13500.0, "s_max": 19000.0},
    "C06": {"exp": 16, "tier": "Lead / Executive (15+ yrs)", "edu": "Bachelor's Degree", "s_min": 13500.0, "s_max": 19000.0},
    "C07": {"exp": 16, "tier": "Lead / Executive (15+ yrs)", "edu": "Doctorate", "s_min": 13500.0, "s_max": 19000.0},
    "C08": {"exp": 18, "tier": "Lead / Executive (15+ yrs)", "edu": "Bachelor's Degree", "s_min": 13500.0, "s_max": 19000.0},
    "C09": {"exp": 11, "tier": "Senior (8-14 yrs)", "edu": "Bachelor's Degree", "s_min": 8500.0, "s_max": 13000.0},
    "C10": {"exp": 14, "tier": "Senior (8-14 yrs)", "edu": "Master's Degree", "s_min": 9500.0, "s_max": 14000.0},
    "C11": {"exp": 5, "tier": "Mid-Level (4-7 yrs)", "edu": "Bachelor's Degree", "s_min": 5500.0, "s_max": 8000.0},
    "C12": {"exp": 20, "tier": "Lead / Executive (15+ yrs)", "edu": "Master's Degree", "s_min": 13500.0, "s_max": 19000.0},
    "C13": {"exp": 27, "tier": "Lead / Executive (15+ yrs)", "edu": "Master's Degree", "s_min": 13500.0, "s_max": 19000.0},
    "C14": {"exp": 7, "tier": "Mid-Level (4-7 yrs)", "edu": "Bachelor's Degree", "s_min": 5500.0, "s_max": 8000.0},
    "C15": {"exp": 6, "tier": "Mid-Level (4-7 yrs)", "edu": "Diploma", "s_min": 5000.0, "s_max": 7500.0},
    "C16": {"exp": 5, "tier": "Mid-Level (4-7 yrs)", "edu": "Bachelor's Degree", "s_min": 5000.0, "s_max": 7500.0},
    "C17": {"exp": 10, "tier": "Senior (8-14 yrs)", "edu": "Associate", "s_min": 8500.0, "s_max": 13000.0},
    "C18": {"exp": 14, "tier": "Senior (8-14 yrs)", "edu": "Associate", "s_min": 8500.0, "s_max": 13000.0},
    "C19": {"exp": 11, "tier": "Senior (8-14 yrs)", "edu": "Bachelor's Degree", "s_min": 8500.0, "s_max": 13000.0},
    "C20": {"exp": 14, "tier": "Senior (8-14 yrs)", "edu": "Bachelor's Degree", "s_min": 8500.0, "s_max": 13000.0},
    # Accounting Candidates (C21 - C30)
    "C21": {"exp": 11, "tier": "Senior (8-14 yrs)", "edu": "Bachelor's Degree", "s_min": 8000.0, "s_max": 12000.0},
    "C22": {"exp": 20, "tier": "Lead / Executive (15+ yrs)", "edu": "Master's Degree", "s_min": 12500.0, "s_max": 18000.0},
    "C23": {"exp": 9, "tier": "Senior (8-14 yrs)", "edu": "Master's Degree", "s_min": 8000.0, "s_max": 12000.0},
    "C24": {"exp": 14, "tier": "Senior (8-14 yrs)", "edu": "Bachelor's Degree", "s_min": 8000.0, "s_max": 12000.0},
    "C25": {"exp": 15, "tier": "Lead / Executive (15+ yrs)", "edu": "Bachelor's Degree", "s_min": 12500.0, "s_max": 18000.0},
    "C26": {"exp": 16, "tier": "Lead / Executive (15+ yrs)", "edu": "Professional Certification", "s_min": 12500.0, "s_max": 18000.0},
    "C27": {"exp": 26, "tier": "Lead / Executive (15+ yrs)", "edu": "Bachelor's Degree", "s_min": 12500.0, "s_max": 18000.0},
    "C28": {"exp": 16, "tier": "Lead / Executive (15+ yrs)", "edu": "Bachelor's Degree", "s_min": 12500.0, "s_max": 18000.0},
    "C29": {"exp": 5, "tier": "Mid-Level (4-7 yrs)", "edu": "Bachelor's Degree", "s_min": 5000.0, "s_max": 7500.0},
    "C30": {"exp": 26, "tier": "Lead / Executive (15+ yrs)", "edu": "Diploma", "s_min": 12500.0, "s_max": 18000.0},
}

malaysia_locations_it = [
    "Kuala Lumpur", "Petaling Jaya, Selangor", "Cyberjaya, Selangor", "Kuala Lumpur",
    "Shah Alam, Selangor", "Penang", "Kuala Lumpur", "Subang Jaya, Selangor",
    "Johor Bahru", "Kuala Lumpur", "Petaling Jaya, Selangor", "Kuala Lumpur",
    "Cyberjaya, Selangor", "Penang", "Kuala Lumpur", "Shah Alam, Selangor",
    "Petaling Jaya, Selangor", "Kuala Lumpur", "Johor Bahru", "Open to any (Malaysia)"
]

it_target_roles = [
    "Information Security Specialist / Cybersecurity Engineer",
    "Information Technology Specialist / Systems Engineer",
    "IT Manager / Senior Systems Analyst",
    "IT Infrastructure Manager",
    "Director of Information Technology / Head of IT",
    "IT Consultant / Solutions Architect",
    "IT Operations Director / Senior IT Manager",
    "IT Project Manager",
    "Senior IT Advisory Consultant",
    "IT Manager",
    "IT Coordinator / Technical Support Lead",
    "IT Support Specialist / Systems Administrator",
    "Enterprise IT Director / Technology Head",
    "Senior Systems Specialist / Infrastructure Engineer",
    "IT Operations Coordinator / Administrator",
    "IT Support Technician / Desktop Support Engineer",
    "IT Director / Technology Operations Head",
    "Senior Network & Systems Specialist",
    "Network Support Specialist / Systems Engineer",
    "Senior System Administrator / Linux Engineer"
]

malaysia_locations_acc = [
    "Kuala Lumpur", "Petaling Jaya, Selangor", "Kuala Lumpur", "Penang",
    "Shah Alam, Selangor", "Kuala Lumpur", "Johor Bahru", "Petaling Jaya, Selangor",
    "Kuala Lumpur", "Open to any (Malaysia)"
]

acc_target_roles = [
    "Staff Accountant / Financial Analyst",
    "Payroll Specialist / Senior Payroll Accountant",
    "Financial Accountant",
    "Accountant / Accounts Executive",
    "Senior Accountant / Finance Lead",
    "Project Accountant",
    "Staff Accountant / General Ledger Executive",
    "Accountant / Financial Reporting Executive",
    "General Accountant",
    "Corporate Accountant / Senior Finance Manager"
]

candidate_records = []

# Build IT candidate profiles (C01 - C20)
for idx, row in it_sampled.iterrows():
    cid = f"C{idx+1:02d}"
    text = str(row["Resume_str"]).strip()
    header = re.sub(r'\s+', ' ', text.split('\n')[0].strip())
    skills = extract_skills(text, "INFORMATION-TECHNOLOGY")
    
    aud = audited_candidate_profiles[cid]

    candidate_records.append({
        "candidate_id": cid,
        "resume_id": row["ID"],
        "category": "INFORMATION-TECHNOLOGY",
        "original_header": header[:60],
        "detected_experience_years": aud["exp"],
        "experience_tier": aud["tier"],
        "detected_education": aud["edu"],
        "extracted_skills": skills,
        "preferred_job_title": it_target_roles[idx],
        "preferred_location": malaysia_locations_it[idx],
        "expected_salary_min": aud["s_min"],
        "expected_salary_max": aud["s_max"],
        "preferred_job_type": "Contract/Temp" if idx in [7, 15] else "Full time",
        "word_count": len(text.split()),
        "clean_resume_text": text
    })

# Build Accounting candidate profiles (C21 - C30)
for idx, row in acc_sampled.iterrows():
    cid = f"C{idx+21:02d}"
    text = str(row["Resume_str"]).strip()
    header = re.sub(r'\s+', ' ', text.split('\n')[0].strip())
    skills = extract_skills(text, "ACCOUNTANT")
    
    aud = audited_candidate_profiles[cid]

    candidate_records.append({
        "candidate_id": cid,
        "resume_id": row["ID"],
        "category": "ACCOUNTANT",
        "original_header": header[:60],
        "detected_experience_years": aud["exp"],
        "experience_tier": aud["tier"],
        "detected_education": aud["edu"],
        "extracted_skills": skills,
        "preferred_job_title": acc_target_roles[idx],
        "preferred_location": malaysia_locations_acc[idx],
        "expected_salary_min": aud["s_min"],
        "expected_salary_max": aud["s_max"],
        "preferred_job_type": "Contract/Temp" if idx == 5 else "Full time",
        "word_count": len(text.split()),
        "clean_resume_text": text
    })

benchmark_resumes_df = pd.DataFrame(candidate_records)
print(f"Processed {len(benchmark_resumes_df)} candidate profiles.")
print("\nExperience level distribution:")
print(benchmark_resumes_df['experience_tier'].value_counts().to_string())
print("\nEducation distribution:")
print(benchmark_resumes_df['detected_education'].value_counts().to_string())
print("\nSample candidate profiles (first 5):")
display_cols = ["candidate_id", "category", "preferred_job_title", "experience_tier", "detected_education", "expected_salary_min", "expected_salary_max", "preferred_location", "preferred_job_type"]
print(benchmark_resumes_df[display_cols].head(5).to_string(index=False))


Successfully processed 30 benchmark candidate profiles.

Audited Experience Tier Distribution:
experience_tier
Lead / Executive (15+ yrs)    14
Senior (8-14 yrs)             10
Mid-Level (4-7 yrs)            6

Audited Education Distribution:
detected_education
Bachelor's Degree             16
Master's Degree                7
Associate                      3
Diploma                        2
Doctorate                      1
Professional Certification     1

Sample Candidate Profile Overview (First 5 candidates):
candidate_id               category                                      preferred_job_title            experience_tier detected_education  expected_salary_min  expected_salary_max      preferred_location preferred_job_type
         C01 INFORMATION-TECHNOLOGY Information Security Specialist / Cybersecurity Engineer        Mid-Level (4-7 yrs)          Associate               5500.0               8000.0            Kuala Lumpur          Full time
         C02 INFORMATION-TECHNOLOGY

### 2. JobStreet Data Cleaning and Sampling

Preprocessing steps:
1. **Deduplication**: Remove duplicate postings with the same title, company, and location.
2. **Length filtering**: Filter out jobs with descriptions under 30 words so there are enough details to evaluate.
3. **Salary parsing**:
   - Clean whitespace and encoding characters (`\xa0`).
   - Extract min and max salary from formats like `RM 3,000 - RM 5,000` or `MYR 4,000`.
   - Keep track of `is_salary_disclosed` so we can test both disclosed and undisclosed salary cases.
4. **Sampling 60 jobs**:
   - **40 IT jobs (`J01` to `J40`)**: 20 with disclosed salary and 20 undisclosed.
   - **20 Accounting jobs (`J41` to `J60`)**: 10 with disclosed salary and 10 undisclosed.


In [ ]:
# Load JobStreet dataset
jobs_raw_df = pd.read_csv(JOB_CSV_PATH)
print(f"Total raw JobStreet postings: {len(jobs_raw_df):,d}")

# 1. Remove duplicate postings (job_title, company, location)
jobs_dedup_df = jobs_raw_df.drop_duplicates(subset=["job_title", "company", "location"]).copy()
print(f"After removing duplicates: {len(jobs_dedup_df):,d}")

# 2. Filter out short descriptions (< 30 words)
jobs_dedup_df["desc_word_count"] = jobs_dedup_df["descriptions"].apply(lambda x: len(str(x).split()))
jobs_valid_df = jobs_dedup_df[jobs_dedup_df["desc_word_count"] >= 30].copy()
print(f"Postings with >= 30 words: {len(jobs_valid_df):,d}")

# 3. Regex parser for salary in MYR
def parse_malaysian_salary(raw_val):
    if pd.isna(raw_val):
        return None, None, False
    s = str(raw_val).replace('\xa0', ' ').strip()
    
    m = re.search(r'(?:RM|MYR)\s*([\d,]+)\s*(?:-|–|—|to|[^\w\d\s])\s*(?:RM|MYR)?\s*([\d,]+)', s, re.I)
    if m:
        try:
            s_min = float(m.group(1).replace(',', ''))
            s_max = float(m.group(2).replace(',', ''))
            if s_min > s_max:
                s_min, s_max = s_max, s_min
            return s_min, s_max, True
        except:
            pass
            
    m2 = re.search(r'(?:RM|MYR)\s*([\d,]+)', s, re.I)
    if m2:
        try:
            amt = float(m2.group(1).replace(',', ''))
            return amt, amt, True
        except:
            pass
            
    return None, None, False

parsed_salaries = [parse_malaysian_salary(x) for x in jobs_valid_df["salary"]]
jobs_valid_df["salary_min"] = [p[0] for p in parsed_salaries]
jobs_valid_df["salary_max"] = [p[1] for p in parsed_salaries]
jobs_valid_df["is_salary_disclosed"] = [p[2] for p in parsed_salaries]

# Split into IT and Accounting 
it_jobs_pool = jobs_valid_df[jobs_valid_df["category"] == "Information & Communication Technology"].copy()
acc_jobs_pool = jobs_valid_df[jobs_valid_df["category"] == "Accounting"].copy()

print(f"Available IT jobs: {len(it_jobs_pool):,d} ({it_jobs_pool['is_salary_disclosed'].mean()*100:.1f}% have salary)")
print(f"Available Accounting jobs: {len(acc_jobs_pool):,d} ({acc_jobs_pool['is_salary_disclosed'].mean()*100:.1f}% have salary)")

# 4. Sample jobs (50% disclosed salary, 50% undisclosed)
it_disc = it_jobs_pool[it_jobs_pool["is_salary_disclosed"]].sample(n=20, random_state=RANDOM_STATE)
it_undisc = it_jobs_pool[~it_jobs_pool["is_salary_disclosed"]].sample(n=20, random_state=RANDOM_STATE)
it_jobs_sample = pd.concat([it_disc, it_undisc]).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
it_jobs_sample["job_id_code"] = [f"J{i:02d}" for i in range(1, 41)]

acc_disc = acc_jobs_pool[acc_jobs_pool["is_salary_disclosed"]].sample(n=10, random_state=RANDOM_STATE)
acc_undisc = acc_jobs_pool[~acc_jobs_pool["is_salary_disclosed"]].sample(n=10, random_state=RANDOM_STATE)
acc_jobs_sample = pd.concat([acc_disc, acc_undisc]).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
acc_jobs_sample["job_id_code"] = [f"J{i:02d}" for i in range(41, 61)]

benchmark_jobs_df = pd.concat([it_jobs_sample, acc_jobs_sample]).reset_index(drop=True)

job_final_cols = {
    "job_id_code": "job_id_code",
    "job_id": "job_id",
    "job_title": "job_title",
    "company": "company",
    "location": "location",
    "category": "category",
    "subcategory": "subcategory",
    "role": "role",
    "type": "job_type",
    "salary": "salary_raw",
    "salary_min": "salary_min",
    "salary_max": "salary_max",
    "is_salary_disclosed": "is_salary_disclosed",
    "descriptions": "descriptions",
    "desc_word_count": "word_count"
}
benchmark_jobs_df = benchmark_jobs_df[list(job_final_cols.keys())].rename(columns=job_final_cols)

print(f"\nSampled {len(benchmark_jobs_df)} jobs ({len(it_jobs_sample)} IT + {len(acc_jobs_sample)} Accounting).")
print("\nSample job postings (first 5):")
job_display_cols = ["job_id_code", "category", "job_title", "company", "location", "is_salary_disclosed", "salary_min", "salary_max", "job_type"]
print(benchmark_jobs_df[job_display_cols].head(5).to_string(index=False))


Total raw JobStreet postings: 59,306
Postings after deduplication: 54,551
Postings with descriptions >= 30 words: 54,213
Clean IT jobs available: 7,178 (Disclosed salary: 29.1%)
Clean Accounting jobs available: 8,959 (Disclosed salary: 48.2%)

Successfully sampled 60 benchmark jobs (40 IT + 20 Accounting).

Sample Job Postings Overview (First 5 jobs):
job_id_code                               category                    job_title                              company             location  is_salary_disclosed  salary_min  salary_max  job_type
        J01 Information & Communication Technology Java Microservices Developer               UST Global (M) Sdn Bhd             Petaling                 True     10000.0     15000.0 Full time
        J02 Information & Communication Technology                 IT Executive Perodua Global Manufacturing Sdn Bhd      Gombak District                 True      3000.0      3500.0 Full time
        J03 Information & Communication Technology        Applicati

### 3. Export Processed Datasets

Export the processed files to `experiments/dataset/processed/`:
1. `benchmark_resumes.csv`: 30 candidates (20 IT, 10 Accounting) with candidate attributes and search preferences.
2. `benchmark_jobs.csv`: 60 job postings (40 IT, 20 Accounting) with parsed salary and clean text.
3. `sampling_summary.csv`: Overview table comparing sample counts with the original dataset.


In [5]:
# Output paths
resumes_export_path = PROCESSED_DIR / "benchmark_resumes.csv"
jobs_export_path = PROCESSED_DIR / "benchmark_jobs.csv"
summary_export_path = PROCESSED_DIR / "sampling_summary.csv"

# 1. Export resumes
benchmark_resumes_df.to_csv(resumes_export_path, index=False, encoding="utf-8")
print(f"Saved: {resumes_export_path} ({resumes_export_path.stat().st_size / 1024:.1f} KB)")

# 2. Export jobs
benchmark_jobs_df.to_csv(jobs_export_path, index=False, encoding="utf-8")
print(f"Saved: {jobs_export_path} ({jobs_export_path.stat().st_size / 1024:.1f} KB)")

# 3. Export summary table
summary_data = [
    {
        "domain": "Information Technology",
        "sample_type": "Candidates (Resumes)",
        "count": len(benchmark_resumes_df[benchmark_resumes_df["category"] == "INFORMATION-TECHNOLOGY"]),
        "population_available": len(it_resumes_pool),
        "sampling_ratio": f"{len(benchmark_resumes_df[benchmark_resumes_df['category'] == 'INFORMATION-TECHNOLOGY']) / len(it_resumes_pool) * 100:.1f}%",
        "avg_words": round(benchmark_resumes_df[benchmark_resumes_df["category"] == "INFORMATION-TECHNOLOGY"]["word_count"].mean(), 1),
        "disclosed_salary_pct": "100.0% (Simulated Expected)"
    },
    {
        "domain": "Information Technology",
        "sample_type": "Job Postings",
        "count": len(benchmark_jobs_df[benchmark_jobs_df["category"] == "Information & Communication Technology"]),
        "population_available": len(it_jobs_pool),
        "sampling_ratio": f"{len(benchmark_jobs_df[benchmark_jobs_df['category'] == 'Information & Communication Technology']) / len(it_jobs_pool) * 100:.2f}%",
        "avg_words": round(benchmark_jobs_df[benchmark_jobs_df["category"] == "Information & Communication Technology"]["word_count"].mean(), 1),
        "disclosed_salary_pct": f"{benchmark_jobs_df[benchmark_jobs_df['category'] == 'Information & Communication Technology']['is_salary_disclosed'].mean()*100:.1f}%"
    },
    {
        "domain": "Accounting & Finance",
        "sample_type": "Candidates (Resumes)",
        "count": len(benchmark_resumes_df[benchmark_resumes_df["category"] == "ACCOUNTANT"]),
        "population_available": len(acc_resumes_pool),
        "sampling_ratio": f"{len(benchmark_resumes_df[benchmark_resumes_df['category'] == 'ACCOUNTANT']) / len(acc_resumes_pool) * 100:.1f}%",
        "avg_words": round(benchmark_resumes_df[benchmark_resumes_df["category"] == "ACCOUNTANT"]["word_count"].mean(), 1),
        "disclosed_salary_pct": "100.0% (Simulated Expected)"
    },
    {
        "domain": "Accounting & Finance",
        "sample_type": "Job Postings",
        "count": len(benchmark_jobs_df[benchmark_jobs_df["category"] == "Accounting"]),
        "population_available": len(acc_jobs_pool),
        "sampling_ratio": f"{len(benchmark_jobs_df[benchmark_jobs_df['category'] == 'Accounting']) / len(acc_jobs_pool) * 100:.2f}%",
        "avg_words": round(benchmark_jobs_df[benchmark_jobs_df["category"] == "Accounting"]["word_count"].mean(), 1),
        "disclosed_salary_pct": f"{benchmark_jobs_df[benchmark_jobs_df['category'] == 'Accounting']['is_salary_disclosed'].mean()*100:.1f}%"
    }
]

sampling_summary_df = pd.DataFrame(summary_data)
sampling_summary_df.to_csv(summary_export_path, index=False, encoding="utf-8")
print(f"Saved: {summary_export_path} ({summary_export_path.stat().st_size} bytes)")

# Check that exported files have expected row counts
check_res = pd.read_csv(resumes_export_path)
check_jobs = pd.read_csv(jobs_export_path)
assert len(check_res) == 30, f"Expected 30 resumes, got {len(check_res)}"
assert len(check_jobs) == 60, f"Expected 60 jobs, got {len(check_jobs)}"
print("\nVerification passed: All datasets exported with expected row counts.")


Saved: c:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\processed\benchmark_resumes.csv (186.4 KB)
Saved: c:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\processed\benchmark_jobs.csv (126.4 KB)
Saved: c:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\processed\sampling_summary.csv (398 bytes)

Export Integrity Verified: All benchmark datasets match target dimensions.


### Check the distributions of the sampled candidates and jobs


In [6]:
print("-" * 70)
print("BENCHMARK TEST SUITE DIAGNOSTICS & VERIFICATION")
print("-" * 70)

print("\n1. CANDIDATE QUALIFICATION & SENIORITY DISTRIBUTION:")
seniority_dist = benchmark_resumes_df.groupby(["category", "experience_tier"]).size().unstack(fill_value=0)
print(seniority_dist)

print("\n2. CANDIDATE EDUCATION LEVEL DISTRIBUTION:")
edu_dist = benchmark_resumes_df.groupby(["category", "detected_education"]).size().unstack(fill_value=0)
print(edu_dist)

print("\n3. CANDIDATE SALARY EXPECTATIONS (RM):")
cand_sal = benchmark_resumes_df.groupby("category")[["expected_salary_min", "expected_salary_max"]].agg(["mean", "min", "max"])
print(cand_sal)

print("\n4. JOB SALARY OFFERINGS (RM - Disclosed Postings Only):")
job_sal = benchmark_jobs_df[benchmark_jobs_df["is_salary_disclosed"]].groupby("category")[["salary_min", "salary_max"]].agg(["mean", "min", "max"])
print(job_sal)

print("\n5. TOP JOB SUB-DISCIPLINES IN BENCHMARK:")
print(benchmark_jobs_df.groupby(["category", "subcategory"]).size().rename("count").reset_index().to_string(index=False))

print("\n6. LOCATION DISTRIBUTION:")
print("Candidate Preferred Locations:")
print(benchmark_resumes_df["preferred_location"].value_counts().head(6).to_string())
print("\nJob Street Posting Locations:")
print(benchmark_jobs_df["location"].value_counts().head(6).to_string())


----------------------------------------------------------------------
BENCHMARK TEST SUITE DIAGNOSTICS & VERIFICATION
----------------------------------------------------------------------

1. CANDIDATE QUALIFICATION & SENIORITY DISTRIBUTION:
experience_tier         Lead / Executive (15+ yrs)  ...  Senior (8-14 yrs)
category                                            ...                   
ACCOUNTANT                                       6  ...                  3
INFORMATION-TECHNOLOGY                           8  ...                  7

[2 rows x 3 columns]

2. CANDIDATE EDUCATION LEVEL DISTRIBUTION:
detected_education      Associate  ...  Professional Certification
category                           ...                            
ACCOUNTANT                      0  ...                           1
INFORMATION-TECHNOLOGY          3  ...                           0

[2 rows x 6 columns]

3. CANDIDATE SALARY EXPECTATIONS (RM):
                       expected_salary_min          ... expe